In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

from lazypredict.Supervised import LazyClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [15]:
df = pd.read_csv("data/Churn Sampled.csv")

df = df[["Geography", "Gender","Card Type","Age","CreditScore","Tenure","Point Earned","Balance","EstimatedSalary", 
    "NumOfProducts","IsActiveMember","HasCrCard","Satisfaction Score","Exited"]]

df.sample(10)

,Geography,Gender,Card Type,Age,CreditScore,Tenure,Point Earned,Balance,EstimatedSalary,NumOfProducts,IsActiveMember,HasCrCard,Satisfaction Score,Exited
2624,Germany,Female,SILVER,57,351,4,453,163146.46,169621.69,1,0,1,4,1
684,France,Male,SILVER,40,645,6,323,131411.24,194656.11,1,1,1,4,0
1381,France,Female,GOLD,35,479,2,889,113090.40,195649.79,1,0,1,4,0
2201,France,Female,DIAMOND,58,570,8,257,0.00,116503.92,1,1,0,2,1
101,France,Female,SILVER,34,672,8,923,0.00,16245.25,2,1,1,5,0
390,Spain,Female,PLATINUM,29,636,6,332,157576.47,101102.39,2,1,1,1,0
988,France,Female,DIAMOND,51,446,4,731,105056.13,70613.52,1,0,0,2,0
3523,Germany,Female,DIAMOND,48,754,7,671,141819.02,93550.53,1,0,1,5,1
3091,Spain,Female,PLATINUM,38,619,1,989,0.00,112442.63,1,0,1,4,1
3377,Germany,Male,SILVER,41,632,8,658,127205.32,93874.87,4,0,1,1,1


In [16]:
X = df.drop("Exited", axis = 1)
y = df["Exited"]

In [17]:
xtrain, xtest, ytrain, ytest = train_test_split(X, y, train_size = 0.8, stratify = y, random_state = 42)

In [18]:
xtrain.shape,xtest.shape, ytrain.shape, ytest.shape

((3310, 13), (828, 13), (3310,), (828,))

In [19]:
xtrain, xval, ytrain, yval = train_test_split(xtrain, ytrain, train_size = 0.85, 
                                            stratify = ytrain, random_state = 42)

In [20]:
xtrain.shape, xval.shape, ytrain.shape, yval.shape 

((2813, 13), (497, 13), (2813,), (497,))

In [21]:
for col in df.select_dtypes(include= "object").columns:
    print(col, ":",df[col].unique())

Geography : ['France' 'Germany' 'Spain']
Gender : ['Male' 'Female']
Card Type : ['GOLD' 'PLATINUM' 'SILVER' 'DIAMOND']


In [22]:
le_geography = LabelEncoder()
le_gender = LabelEncoder()
oe_crdtype = OrdinalEncoder(categories = [["SILVER", "GOLD", "DIAMOND", "PLATINUM"]])

# Geography
xtrain["Geography"] = le_geography.fit_transform(xtrain["Geography"])
xval["Geography"] = le_geography.transform(xval["Geography"])
xtest["Geography"] = le_geography.transform(xtest["Geography"])

# Gender
xtrain["Gender"] = le_gender.fit_transform(xtrain["Gender"])
xval["Gender"] = le_gender.transform(xval["Gender"])
xtest["Gender"] = le_gender.transform(xtest["Gender"])

# Card Type
xtrain["Card Type"] = oe_crdtype.fit_transform(xtrain[["Card Type"]])
xval["Card Type"] = oe_crdtype.transform(xval[["Card Type"]])
xtest["Card Type"] = oe_crdtype.transform(xtest[["Card Type"]])

In [23]:
# Age, CreditScore, Point Earned, Balance, EstimatedSalary
mms = MinMaxScaler()
xtrain[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
            = mms.fit_transform(xtrain[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])
xval[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
            = mms.transform(xval[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])
xtest[["Age", "CreditScore", "Point Earned", "Balance", "EstimatedSalary"]] \
            = mms.transform(xtest[["Age","CreditScore", "Point Earned", "Balance", "EstimatedSalary"]])

In [24]:
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)

In [25]:
models, predictions = clf.fit(xtrain, xval, ytrain, yval)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 1385, number of negative: 1428
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000204 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1120
[LightGBM] [Info] Number of data points in the train set: 2813, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.492357 -> initscore=-0.030575
[LightGBM] [Info] Start training from score -0.030575


In [26]:
models

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
RandomForestClassifier,0.78,0.78,0.78,0.78,0.77
ExtraTreesClassifier,0.77,0.77,0.77,0.77,0.58
AdaBoostClassifier,0.76,0.76,0.76,0.76,0.40
SVC,0.75,0.75,0.75,0.75,0.53
LGBMClassifier,0.75,0.75,0.75,0.75,2.74
NuSVC,0.74,0.74,0.74,0.74,0.62
GaussianNB,0.73,0.73,0.73,0.73,0.03
XGBClassifier,0.73,0.73,0.73,0.73,0.24
BaggingClassifier,0.72,0.72,0.72,0.72,0.22


In [28]:
predictions

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
BaggingClassifier,1.00,1.00,None,1.00,0.05
ExtraTreeClassifier,1.00,1.00,None,1.00,0.02
DecisionTreeClassifier,1.00,1.00,None,1.00,0.02
KNeighborsClassifier,1.00,1.00,None,1.00,0.02
LabelPropagation,1.00,1.00,None,1.00,0.02
GaussianNB,1.00,1.00,None,1.00,0.02
ExtraTreesClassifier,1.00,1.00,None,1.00,0.15
LogisticRegression,1.00,1.00,None,1.00,0.03
LinearSVC,1.00,1.00,None,1.00,0.03


In [34]:
models, predictions = clf.fit(xtrain, xtest, ytrain, ytest)

'tuple' object has no attribute '__name__'
Invalid Classifier(s)


  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 1385, number of negative: 1428
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002119 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1120
[LightGBM] [Info] Number of data points in the train set: 2813, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.492357 -> initscore=-0.030575
[LightGBM] [Info] Start training from score -0.030575


In [35]:
models

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LGBMClassifier,0.78,0.78,0.78,0.78,0.18
RandomForestClassifier,0.77,0.77,0.77,0.77,0.69
AdaBoostClassifier,0.77,0.77,0.77,0.77,0.70
NuSVC,0.76,0.76,0.76,0.76,0.69
SVC,0.76,0.76,0.76,0.76,0.59
ExtraTreesClassifier,0.76,0.76,0.76,0.76,0.51
BaggingClassifier,0.75,0.75,0.75,0.75,0.22
XGBClassifier,0.74,0.74,0.74,0.74,0.23
GaussianNB,0.73,0.73,0.73,0.73,0.03


In [36]:
predictions

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LGBMClassifier,0.78,0.78,0.78,0.78,0.18
RandomForestClassifier,0.77,0.77,0.77,0.77,0.69
AdaBoostClassifier,0.77,0.77,0.77,0.77,0.70
NuSVC,0.76,0.76,0.76,0.76,0.69
SVC,0.76,0.76,0.76,0.76,0.59
ExtraTreesClassifier,0.76,0.76,0.76,0.76,0.51
BaggingClassifier,0.75,0.75,0.75,0.75,0.22
XGBClassifier,0.74,0.74,0.74,0.74,0.23
GaussianNB,0.73,0.73,0.73,0.73,0.03


In [37]:
models, predictions = clf.fit(xtrain, xval, xtest, ytrain, yval, ytest)

TypeError: LazyClassifier.fit() takes 5 positional arguments but 7 were given

In [38]:
help(clf)

Help on LazyClassifier in module lazypredict.Supervised object:

class LazyClassifier(builtins.object)
 |  LazyClassifier(
 |      verbose=0,
 |      ignore_warnings=True,
 |      custom_metric=None,
 |      predictions=False,
 |      random_state=42,
 |      classifiers='all'
 |  )
 |
 |  This module helps in fitting to all the classification algorithms that are available in Scikit-learn
 |  Parameters
 |  ----------
 |  verbose : int, optional (default=0)
 |      For the liblinear and lbfgs solvers set verbose to any positive
 |      number for verbosity.
 |  ignore_warnings : bool, optional (default=True)
 |      When set to True, the warning related to algorigms that are not able to run are ignored.
 |  custom_metric : function, optional (default=None)
 |      When function is provided, models are evaluated based on the custom evaluation metric provided.
 |  prediction : bool, optional (default=False)
 |      When set to True, the predictions of all the models models are returned a